# Runde 2 · Sitzung r2_pc (Welle 2): Predictive Coding gegen Backprop (C-T7) und Kreuzpaare mit PC-Netzen (AT5)

C-T7 bei W 128 und 256: PC-Seite auf den PC-µP-Netzen und BP-Seite auf den BP-µP-Netzen der Runde 1 (T4b und der PC-gepaarte Zwilling von C-T5), unabhängig von den Freigaben (G-meta und G-PC entscheidet r2_ana); dazu die AT5-Paare mit einem PC-Netz bei jeder PC-Breite, mit den Backprop-Netzen aus r2_t1.

**Ausführen.** *Laufzeit → Laufzeittyp ändern → A100 GPU*, dann *Laufzeit → Alle ausführen*. Nur eine A100 ist zulässig
(Freigabe G2). Erwartete Dauer (Plan 7.2): Arbeit 6,0 min, mit Einrichtung 9,0 min, mit Reserve 12,0 min.

**Zeitgrenze.** 20 Minuten nach der ersten Zelle hält der Lauf nach dem gerade laufenden Schritt an (Plan 7.4) und schreibt
`session_partial.json`. Dann dieses Notebook in einer neuen Colab-Sitzung noch einmal ganz ausführen: es läuft als
Fortsetzungssitzung (`r2_pcb`) weiter, fertige Schritte werden übersprungen. Ebenso nach einem Verbindungsabbruch.

**Startbedingungen (Plan 7.3).** Welle 2: startet erst, wenn `dsse_runs/r2_t1/session_done.json` existiert und die PC-Ausgaben aller sechs Runde-1-Teile endgültig sind (Freigabe G5: `shard_done.json`, sonst der Eintrag in `r1_finality.json`).

**Ordner.** Liest alle sechs Runde-1-Ordner `dsse_runs/gpu1` … `gpu6` (beide Konten) und aus Runde 2 `dsse_runs/r2_t1`.

**Zwei Konten.** Die Runde-1-Teile `gpu1`–`gpu3` liefen in einem Google-Konto (A), `gpu4`–`gpu6` in einem anderen (B). Einen Ordner aus dem anderen Konto vorher dort freigeben (Leserecht genügt) und hier in *Meine Ablage › dsse_runs* über *Verknüpfung zu Drive hinzufügen* einhängen, oder ihn in dieses Konto nach `dsse_runs/` kopieren. Der Runner liest diese Ordner nur und schreibt ausschließlich nach `dsse_runs/r2_pc`.

**Runde-1-Endstand (R1.final).** Fehlt in einem Runde-1-Ordner auch nach zwei Läufen seines Notebooks `shard_done.json` (PC-Jobs fehlen) oder eine Blockzusammenfassung, trägt der Betreiber das von Hand in `dsse_runs/r1_finality.json` ein (kein Skript schreibt diese Datei; Plan 2.1), zum Beispiel
```json
{"gpu4": {"runs": [{"start": "2026-11-12T15:02Z", "end": "2026-11-12T15:24Z"},
                  {"start": "2026-11-12T16:10Z", "end": "2026-11-12T16:31Z"}], "final": true}}
```
Ab zwei eingetragenen Läufen und `"final": true` gilt der Teil als endgültig; Teile an einem Block ohne Zusammenfassung laufen dann nicht ('not run (round-1 output missing)').

**Selbsttest.** Vor dem Hauptlauf laufen die eigenen Tests der Skripte (r2_toy_common.py selftest mit den W0-Äquivalenztests und dem Referenzvergleich) und dann die ganze Sitzung im
MOCK/QUICK-Modus (winziges Zufallsmodell, Kurzläufe) auf einem Schein-`dsse_runs`, das die Runde-1-Skripte erzeugen – nur Code, keine
Ergebnisse; einmal je Code-Stand. Er kostet einige Minuten der Zeitgrenze: am besten das Notebook zuerst einmal mit
`NUR_SELBSTTEST = True` ausführen; der echte Lauf überspringt ihn dann.

**Ergebnisse** in Google Drive unter `dsse_runs/r2_pc` (Manifest `manifest_r2_pc.json`, `session_done.json` mit den Schrittzeiten).

Schritte: `t7` (r2_pc_toy.py t7) · `cross_pc` (r2_pc_toy.py cross_pc)

Modell `Qwen/Qwen3-1.7B` (float32, TF32 aus) in fester Revision mit Prüfsummen.
Code: `erikiss/DSSE`, Branch `ccr-6fe02b21-est040`, Ordner `analysis/ju_space/round2`. Plan: eingefroren mit Commit `5b46e38cd800`
(`results/ju_space/plans/abschnitt11_12_runde2_plan.md`, sha256 `5e476340dfdb1d82…`); der Runner läuft mit keiner
anderen Planfassung (Freigabe G1).

In [ ]:
T0 = __import__("time").time()     # Beginn der Sitzung: die Zeitgrenze von 20 min (Plan 7.4) zählt ab hier
SESSION, KIND = "r2_pc", "toy"
BRANCH = "ccr-6fe02b21-est040"
MODEL_REV = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"      # Qwen/Qwen3-1.7B (wie Runde 1; Prüfsummen in analysis/ju_space/gpu/data/checksums.json)
PERSONA_REV = "d533f626cc321c92175a58ee570aa3cdb87238d1"    # Anthropic/model-written-evals
BRAUCHT = ['gpu1', 'gpu2', 'gpu3', 'gpu4', 'gpu5', 'gpu6', 'r2_t1']    # Ordner in dsse_runs, die diese Sitzung liest
OPTIONAL = []    # nur Versionsangaben; dürfen fehlen
NICHT_GELAUFEN = []          # nur r2_ana: Sitzungen, die nach zwei Versuchen als 'not run' gelten (z. B. ["r2_t4"])
SELBSTTEST = True            # MOCK/QUICK-Selbsttest vor dem Hauptlauf (einmal je Code-Stand)
NUR_SELBSTTEST = False       # True: nur einrichten und testen, kein Hauptlauf
import os, sys, json, shutil, subprocess, hashlib, lzma, time
import torch
assert torch.cuda.is_available(), "Keine GPU: Laufzeit → Laufzeittyp ändern → A100 GPU, dann neu starten."
GPU = torch.cuda.get_device_name(0); GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU}, {GB:.0f} GB, torch {torch.__version__}")
assert "A100" in GPU, f"Freigabe G2: nur eine A100 ist zulässig (hier {GPU}). Laufzeit → Laufzeittyp ändern → A100 GPU."

In [ ]:
# Ablage: Google Drive (dsse_runs/<Sitzung>), Arbeitskopie auf der lokalen Platte; Fortsetzung aus Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/dsse_runs"
    DRIVE = f"{RUNS}/{SESSION}"
except Exception as e:
    print("Kein Google Drive, Ergebnisse nur lokal:", e)
    RUNS = DRIVE = None
WORK, ROOT, REPO = f"/content/work/{SESSION}", "/content/root", "/content/DSSE"
os.makedirs(WORK, exist_ok=True)
if DRIVE and os.path.isdir(DRIVE) and os.listdir(DRIVE):
    shutil.copytree(DRIVE, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.tmp", "*.r2tmp"))
    print("Fortsetzung: bisheriger Stand aus Drive übernommen; fertige Schritte werden übersprungen.")
if RUNS:                                   # nur Ordnernamen, kein Inhalt
    fehlt = [f for f in BRAUCHT if not os.path.isdir(f"{RUNS}/{f}")]
    print("Diese Sitzung liest aus dsse_runs:", ", ".join(BRAUCHT) or "nichts")
    if fehlt:
        print("FEHLT:", ", ".join(fehlt), "- aus dem anderen Konto freigeben und hier verknüpfen oder kopieren; sonst wartet der Runner.")
    if any(not os.path.isdir(f"{RUNS}/{f}") for f in OPTIONAL):
        print("Nicht sichtbar (nur Versionsangaben, darf fehlen):", ", ".join(f for f in OPTIONAL if not os.path.isdir(f"{RUNS}/{f}")))
print("Ergebnisse:", DRIVE or WORK)

In [ ]:
# Pakete und Code: der Ordner analysis/ju_space (mit round2/) und der eingefrorene Plan; ohne --depth, denn der Runner prüft den
# Einfrier-Commit (G1) und den Code der Runde-1-Teile (G3) in der Git-Geschichte (Dateiinhalte werden bei Bedarf nachgeladen)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==5.22.2"], check=True)
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--sparse", "-b", BRANCH, "https://github.com/erikiss/DSSE", REPO], check=True)
subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", "analysis/ju_space", "results/ju_space/plans"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %cd %s"], capture_output=True, text=True).stdout.strip())
A = f"{REPO}/analysis/ju_space"; D = f"{A}/gpu/data"; R2 = f"{A}/round2"
SUMS = json.load(open(f"{D}/checksums.json"))

In [ ]:
# Eingaben: Modelle und Persona-Sätze von Hugging Face (feste Revision), übrige Daten aus dem Repository; alles mit Prüfsumme.
# models/verified.json hält fest, was geprüft wurde (der Runner trägt es ins Manifest ein; ohne sie prüft er selbst).
from huggingface_hub import snapshot_download, hf_hub_download

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()

jetzt = lambda: time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
os.makedirs(f"{ROOT}/models", exist_ok=True)
VERIFIED = json.load(open(f"{ROOT}/models/verified.json")) if os.path.exists(f"{ROOT}/models/verified.json") else {}
QWEN = KIND == "qwen"
MOCK_QWEN = SELBSTTEST and SESSION == "r2_ana"        # der volle Selbsttest von r2_ana braucht nur Konfiguration und Tokenizer
if QWEN or MOCK_QWEN:
    M = f"{ROOT}/models/qwen3-1.7b"
    snapshot_download("Qwen/Qwen3-1.7B", revision=MODEL_REV, local_dir=M,
                      allow_patterns=["*.json", "merges.txt", "vocab.json"] + (["*.safetensors"] if QWEN else []))
    for fn, h in SUMS["model_sha256"].items():
        if QWEN or not fn.endswith(".safetensors"):
            assert sha256(f"{M}/{fn}") == h, f"Prüfsumme stimmt nicht: {fn}"
    if QWEN:
        VERIFIED["qwen3-1.7b"] = dict(ok=True, repo="Qwen/Qwen3-1.7B", revision=MODEL_REV, sha256=SUMS["model_sha256"], checked=jetzt())
    P = f"{ROOT}/rsi_dl/persona"; os.makedirs(P, exist_ok=True)
    for n, h in SUMS["persona_git_sha1"].items():
        data = open(hf_hub_download("Anthropic/model-written-evals", f"persona/{n}.jsonl", repo_type="dataset",
                                    revision=PERSONA_REV), "rb").read()
        assert hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest() == h, f"Prüfsumme stimmt nicht: {n}"
        open(f"{P}/{n}.jsonl", "wb").write(data)
if SESSION == "r2_q3":                                  # Qwen3-0.6B: ein Fehler beendet nichts, der Arm läuft dann nicht (G33)
    M6 = f"{ROOT}/models/qwen3-0.6b"
    try:
        snapshot_download(QWEN06["repo"], revision=QWEN06["revision"], local_dir=M6,
                          allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json"])
        got = {fn: sha256(f"{M6}/{fn}") for fn in QWEN06["sha256"]}
        ok06 = got == QWEN06["sha256"]
    except Exception as e:
        got, ok06 = repr(e), False
    VERIFIED["qwen3-0.6b"] = dict(ok=ok06, repo=QWEN06["repo"], revision=QWEN06["revision"], sha256=got, checked=jetzt())
    print("Qwen3-0.6B geprüft." if ok06 else "Qwen3-0.6B: Download oder Prüfsumme fehlgeschlagen - der 0.6B-Arm läuft nicht (G33).")
for src, dst in SUMS["inputs"].items():
    if not os.path.exists(f"{D}/{src}"):
        print("fehlt im Repository:", src); continue
    os.makedirs(os.path.dirname(f"{ROOT}/{dst['path']}"), exist_ok=True)
    raw = lzma.open(f"{D}/{src}").read() if src.endswith(".xz") else open(f"{D}/{src}", "rb").read()
    open(f"{ROOT}/{dst['path']}", "wb").write(raw)
    assert sha256(f"{ROOT}/{dst['path']}") == dst["sha256"], f"Prüfsumme stimmt nicht: {dst['path']}"
json.dump(VERIFIED, open(f"{ROOT}/models/verified.json", "w"), indent=1)
print("Modelle und Eingaben geprüft:", ", ".join(VERIFIED) or "keine Modelle nötig")

In [ ]:
# Selbsttest (nur Code), einmal je Code-Stand (die Marke liegt danach in dsse_runs/<Sitzung>). Der Runner baut ein Schein-dsse_runs
# aus rk_loop.py MOCK=1 und pc_meta.py QUICK=1 (keine Runde-1-Ausgabe), führt die eigenen Tests der Skripte aus
# (r2_toy_common.py selftest mit den W0-Äquivalenztests und dem Referenzvergleich)
# und dann die ganze Sitzung im MOCK/QUICK-Modus über denselben Code wie der Hauptlauf.
MARKE = f"{WORK}/selftest_ok_{COMMIT[:12]}.json"
if not SELBSTTEST:
    print("Selbsttest ausgelassen (SELBSTTEST = False).")
elif os.path.exists(MARKE):
    print("Selbsttest für diesen Code-Stand schon bestanden:", json.load(open(MARKE)))
else:
    t = time.time(); ST = "/content/r2_selftest"; shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST)
    LOG = f"{ST}/selftest.log"
    with open(LOG, "w") as lg:
        p = subprocess.Popen([sys.executable, "-u", f"{R2}/r2_run_session.py", SESSION, ROOT, ST, "--selftest"],
                             env=dict(os.environ, RUN_DEVICE="cuda", PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            lg.write(line)
            if any(k in line for k in ("[selftest]", "[mock]", "[run]", "FAILED", "SELFTEST")):
                print(line, end="")
    if p.wait():
        print("".join(open(LOG).readlines()[-60:]))
        raise SystemExit(f"Selbsttest fehlgeschlagen (Protokoll {LOG}; die Protokolle der Schritte unter {ST}/work/*/logs).")
    json.dump(dict(commit=COMMIT, seconds=round(time.time() - t), time=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())), open(MARKE, "w"))
    print(f"Selbsttest bestanden ({time.time() - t:.0f}s).")

In [ ]:
# Hauptlauf: Freigaben G1/G2, Startbedingungen, Kopie der Runde-1-Dateien, Manifest, Schritte mit Zeitgrenze und Drive-Abgleich
MELDUNG = {0: "Sitzung vollständig (session_done.json).",
           1: "Schritte fehlgeschlagen - Notebook erneut ausführen; fertige Schritte werden übersprungen, fehlgeschlagene wiederholt.",
           2: "Freigabe G1 (Planfassung) oder G2 (A100) nicht erfüllt, oder eine Modellprüfsumme stimmt nicht - nichts gelaufen.",
           3: "Zeitgrenze erreicht (session_partial.json) - Notebook in einer neuen Sitzung erneut ausführen (Fortsetzung).",
           4: "Startbedingungen nicht erfüllt (start_check.json) - auf Runde 1 bzw. die anderen Sitzungen warten."}
RC = None
if NUR_SELBSTTEST:
    print("NUR_SELBSTTEST = True: kein Hauptlauf.")
else:
    env = dict(os.environ, R2_T0=repr(T0), RUN_DEVICE="cuda", R2_DECLARED_NOT_RUN=",".join(NICHT_GELAUFEN))
    cmd = [sys.executable, "-u", f"{R2}/r2_run_session.py", SESSION, ROOT, WORK] + ([DRIVE] if DRIVE else [])
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    RC = p.wait()
    print(MELDUNG.get(RC, f"Unerwarteter Rückgabewert {RC}; Protokolle in {WORK}/logs"))

In [ ]:
# Übersicht: Schritte mit Zustand und Zeit; Dateien der Sitzung
for name in ("session_done.json", "session_partial.json", "start_check.json"):
    p = f"{WORK}/{name}"
    if not os.path.exists(p):
        continue
    S = json.load(open(p)); print(name)
    if "steps" in S:
        for k, v in S["steps"].items():
            sec = f"{v['seconds']:.0f}s" if isinstance(v.get("seconds"), (int, float)) else ""
            print(f"  {k:<18} {v['state']:<9} {sec:>6}  {v.get('note') or ''}")
        print("  Versuche:", [a.get("attempt") for a in S.get("attempts", [])])
    else:
        print("  ", S.get("result"), *S.get("start", {}).get("waiting", []), sep="\n   ")
    break
for d in sorted(os.listdir(WORK)):
    if d not in ("r1", "r2in", "local", "logs", "steps"):
        p = f"{WORK}/{d}"
        size = sum(os.path.getsize(os.path.join(a, f)) for a, _, fs in os.walk(p) for f in fs) if os.path.isdir(p) else os.path.getsize(p)
        print(f"{d:<34} {size / 1e6:9.2f} MB")

In [ ]:
# Abschluss: letzter Abgleich mit Drive (ohne r1/, r2in/, local/), Drive schreiben und trennen
if DRIVE:
    subprocess.run([sys.executable, f"{R2}/r2_run_session.py", SESSION, ROOT, WORK, DRIVE, "--sync-only"])
    drive.flush_and_unmount()
    print("Fertig. Ergebnisse in Google Drive unter dsse_runs/" + SESSION)
if RC not in (None, 0):
    raise SystemExit(MELDUNG.get(RC, f"Rückgabewert {RC}"))